# Install

# By Khushi Shah

In [1]:
!pip install -q groq

# Imports and API Key

In [2]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

# works from local cloned repo
cwd = Path.cwd()
env_path = cwd / "objection-your-honor" / ".env"
if not env_path.exists():
    env_path = cwd / ".env"

load_dotenv(env_path, override=True)

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError(f"GROQ_API_KEY not found. Looked for .env at: {env_path}")

client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-120b"

VALID_BIASES = {"sunk_cost", "anchoring", "confirmation_bias", "loss_aversion", "none"}

# System prompt

In [3]:
SYSTEM_PROMPT = """You are a cognitive bias classifier. You read a person's journal entry about a decision they are weighing and identify which cognitive bias, if any, is shaping their reasoning.

BIAS DEFINITIONS:
- sunk_cost: Justifying a choice by past, unrecoverable investment (time, money, effort) instead of future outcomes. Looks BACKWARD.
- anchoring: Judging value or reasonableness based on an initial reference number (original price, first offer, first quote) that may be arbitrary or irrelevant.
- confirmation_bias: Favoring evidence that supports an existing belief while ignoring, dismissing, or never seeking contrary evidence.
- loss_aversion: Weighing a potential loss much more heavily than an equal or larger potential gain, often preferring the safe or current option out of fear of losing something. Looks FORWARD.
- none: The reasoning is balanced, weighs trade-offs fairly, or no clear bias is present.

RULES:
1. Choose exactly one primary bias. Do NOT force a bias. If the reasoning is sound, return "none".
2. If a second bias is clearly present, put it in secondary_bias. Otherwise null.
3. trigger_phrases must be EXACT verbatim quotes copied from the entry. Max 3. Empty list if bias is "none".
4. confidence_score is between 0 and 1. Use 0.85+ only when the bias is explicit. Use 0.5-0.7 for ambiguous entries. Use below 0.5 if guessing.
5. reasoning is 1-2 sentences, referencing the trigger phrases.
6. Respond with ONLY a valid JSON object with keys: bias_type, secondary_bias, confidence_score, trigger_phrases, reasoning."""

# Few-shot examples

In [4]:
FEW_SHOT = [
    ("I've already spent six months and about 40k on this coding bootcamp. I hate it and the placement stats look bad, but I can't quit now, that would mean all of it was wasted.",
     {"bias_type": "sunk_cost", "secondary_bias": None, "confidence_score": 0.93,
      "trigger_phrases": ["already spent six months and about 40k", "can't quit now", "all of it was wasted"],
      "reasoning": "The decision to continue rests entirely on past investment rather than the bootcamp's future value, even though the person admits it is not working."}),

    ("We've been together four years. Things haven't been good for a while, but leaving now would make those four years pointless.",
     {"bias_type": "sunk_cost", "secondary_bias": None, "confidence_score": 0.82,
      "trigger_phrases": ["leaving now would make those four years pointless"],
      "reasoning": "Staying is justified by time already invested, not by whether the relationship will be good going forward."}),

    ("The laptop was originally 1,20,000 and now it's 85,000, so it's a massive deal. I should grab it before the price goes back up.",
     {"bias_type": "anchoring", "secondary_bias": None, "confidence_score": 0.9,
      "trigger_phrases": ["originally 1,20,000", "it's a massive deal"],
      "reasoning": "The original price is acting as the reference point for value, so the discount feels big regardless of whether 85,000 is fair for the specs."}),

    ("My first offer was 6 LPA, so when they came back with 7 it felt like a big jump and I think I should just accept.",
     {"bias_type": "anchoring", "secondary_bias": None, "confidence_score": 0.85,
      "trigger_phrases": ["first offer was 6 LPA", "felt like a big jump"],
      "reasoning": "The first offer is setting the baseline, making 7 LPA feel generous without comparing it to market rates."}),

    ("I always knew my manager hates me. Today she skipped my update in the meeting, which proves it. The three times she praised me were just her being polite.",
     {"bias_type": "confirmation_bias", "secondary_bias": None, "confidence_score": 0.94,
      "trigger_phrases": ["I always knew my manager hates me", "which proves it", "just her being polite"],
      "reasoning": "A single ambiguous event is read as proof while contradicting evidence (the praise) is explained away."}),

    ("I'm sure this stock will 10x. I only follow people who are bullish on it, and the bearish analysts clearly don't understand the tech.",
     {"bias_type": "confirmation_bias", "secondary_bias": None, "confidence_score": 0.91,
      "trigger_phrases": ["I only follow people who are bullish on it", "bearish analysts clearly don't understand the tech"],
      "reasoning": "The person filters their information sources to match their belief and dismisses opposing views without engaging them."}),

    ("The new job pays 40% more and the team looks better, but I'd lose my seniority and my stable position here. I can't risk losing what I have, even if the upside is bigger.",
     {"bias_type": "loss_aversion", "secondary_bias": None, "confidence_score": 0.92,
      "trigger_phrases": ["can't risk losing what I have", "even if the upside is bigger"],
      "reasoning": "The person explicitly admits the upside is larger but lets fear of losing current status dominate the decision."}),

    ("I know index funds would grow more, but seeing my savings dip even 5% in a bad month would hurt way more than the gains would feel good, so I'm keeping everything in the FD.",
     {"bias_type": "loss_aversion", "secondary_bias": None, "confidence_score": 0.9,
      "trigger_phrases": ["dip even 5%", "hurt way more than the gains would feel good"],
      "reasoning": "The person describes losses as feeling disproportionately worse than equivalent gains and chooses the safe option because of it."}),

    ("I'm choosing between two flats. One is 2k cheaper but a 40 minute commute, the other is closer. I made a table of rent, commute time, and lease flexibility, and since time matters most to me right now, the closer one wins.",
     {"bias_type": "none", "secondary_bias": None, "confidence_score": 0.9,
      "trigger_phrases": [],
      "reasoning": "The person compares trade-offs against stated priorities and reaches a conclusion without leaning on any single distorting reference."}),

    ("I've already put two years into this startup and I can't risk walking away with nothing. Anyone who says it's failing just doesn't get our vision.",
     {"bias_type": "sunk_cost", "secondary_bias": "confirmation_bias", "confidence_score": 0.8,
      "trigger_phrases": ["already put two years into this startup", "just doesn't get our vision"],
      "reasoning": "The main justification is past investment, and criticism is dismissed rather than evaluated, which points to confirmation bias as a secondary factor."}),
]

# Helper functions

In [5]:
def build_messages(entry: str) -> list:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for example_entry, example_output in FEW_SHOT:
        messages.append({"role": "user", "content": example_entry})
        messages.append({"role": "assistant", "content": json.dumps(example_output)})
    messages.append({"role": "user", "content": entry})
    return messages


def verify_triggers(entry: str, phrases: list) -> list:
    """Keep only phrases that actually appear in the entry (case-insensitive)."""
    entry_lower = entry.lower()
    return [p for p in phrases if p.lower() in entry_lower]

# Classifier

In [6]:
def classify_entry(entry: str, retries: int = 1) -> dict:
    entry = entry.strip()
    if len(entry) < 20:
        return {"bias_type": "none", "secondary_bias": None, "confidence_score": 0.0,
                "trigger_phrases": [], "reasoning": "Entry too short to analyze."}

    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=build_messages(entry),
                temperature=0,
                response_format={"type": "json_object"},
            )
            result = json.loads(response.choices[0].message.content)

            if result.get("bias_type") not in VALID_BIASES:
                raise ValueError("Invalid bias_type")
            if result.get("secondary_bias") not in (VALID_BIASES - {"none"}) | {None}:
                result["secondary_bias"] = None

            claimed = result.get("trigger_phrases", [])
            verified = verify_triggers(entry, claimed)

            # Penalize confidence when quoted phrases are not found in the entry
            model_conf = float(result.get("confidence_score", 0.5))
            if result["bias_type"] != "none" and claimed:
                ratio = len(verified) / len(claimed)
                model_conf = model_conf * (0.5 + 0.5 * ratio)
            elif result["bias_type"] != "none" and not claimed:
                model_conf *= 0.5

            result["trigger_phrases"] = verified
            result["confidence_score"] = round(min(max(model_conf, 0.0), 1.0), 2)
            return result

        except (json.JSONDecodeError, ValueError, KeyError):
            if attempt == retries:
                return {"bias_type": "none", "secondary_bias": None, "confidence_score": 0.0,
                        "trigger_phrases": [], "reasoning": "Classification failed after retry."}

# Test harness

In [7]:
tests = [
    ("I've already invested three years in this degree and quitting now would feel like a waste, even though I don't like the field anymore.", "sunk_cost"),
    ("The jacket was 5000, now it's 2999, so I'm basically stealing it and I have to buy it.", "anchoring"),
    ("Everyone who criticizes my plan is just jealous. The one article that supports me proves I'm right.", "confirmation_bias"),
    ("I got a better offer, but what if the startup shuts down in a year? I'd rather keep my safe job.", "loss_aversion"),
    ("I listed pros and cons for both cities, checked cost of living, and picked the one closer to family.", "none"),
]

correct = 0
for entry, expected in tests:
    out = classify_entry(entry)
    match = out["bias_type"] == expected
    correct += match
    print(f"{'PASS' if match else 'FAIL'} | expected={expected} got={out['bias_type']} conf={out['confidence_score']}")
    print(f"      triggers={out['trigger_phrases']}")

print(f"\n{correct}/{len(tests)} correct")

PASS | expected=sunk_cost got=sunk_cost conf=0.92
      triggers=['already invested three years', 'quitting now would feel like a waste']
PASS | expected=anchoring got=anchoring conf=0.92
      triggers=['The jacket was 5000', "now it's 2999", 'I have to buy it']
PASS | expected=confirmation_bias got=confirmation_bias conf=0.94
      triggers=['Everyone who criticizes my plan is just jealous', "The one article that supports me proves I'm right"]
PASS | expected=loss_aversion got=loss_aversion conf=0.92
      triggers=['what if the startup shuts down in a year', "I'd rather keep my safe job"]
PASS | expected=none got=none conf=0.9
      triggers=[]

5/5 correct


# Try your own entry

In [8]:
entry = "I've already spent two years preparing for this exam, so I can't stop now even though I'm burnt out."
print(json.dumps(classify_entry(entry), indent=2))

{
  "bias_type": "sunk_cost",
  "secondary_bias": null,
  "confidence_score": 0.94,
  "trigger_phrases": [
    "already spent two years preparing for this exam",
    "can't stop now"
  ],
  "reasoning": "The decision to continue is justified solely by the time already invested, ignoring current burnout and future outcomes."
}


----

In [9]:
# By Dilna Joseph

In [10]:
# Rebuttal prompt and strategies

In [11]:
REBUTTAL_SYSTEM = """You are opposing counsel in a cross-examination of someone's decision. You are given a journal entry and the cognitive bias a classifier found in it. Your job is to write a counter-scenario that breaks that specific bias.

RULES:
1. Speak directly to the person using "you".
2. Use their own words or numbers from the entry so it feels specific to them. No generic advice.
3. Follow the strategy given for the detected bias.
4. counter_scenario is 3 to 5 sentences of plain language. No bullet points, no headings.
5. challenge_question is one sharp question they can answer honestly in a sentence or two.
6. Be firm but respectful. Do not insult the person or tell them what to decide. The goal is to make them reconsider.
7. Do not use em dashes.
8. Respond with ONLY a valid JSON object with keys: counter_scenario, challenge_question."""

BIAS_STRATEGY = {
    "sunk_cost": "Reset the scenario. Ask them to imagine starting today with zero time, money or effort already spent. Would they choose this again? Point out that the past cost is gone whether they stay or leave, so only future costs and benefits should count. Mention what staying is costing them from here on.",
    "anchoring": "Remove the reference number. Ask what they would think of the current price or offer if they had never seen the original figure. Suggest judging it against independent benchmarks such as market rates or similar options, and show how a different starting number would have changed their feeling about the same deal.",
    "confirmation_bias": "Argue the opposite side as strongly as possible. Build a scenario where a trusted, informed person disagrees with them and has good evidence. Bring up the evidence they dismissed and treat it seriously. Ask what it would take for them to be wrong.",
    "loss_aversion": "Put the loss and the gain side by side. Describe what they give up by NOT acting, the upside they keep walking away from, and how it looks if they were already holding the new option and were asked to give it up. Shrink the feared loss to its realistic size and mention a way to limit the downside.",
}

FALLBACK_QUESTION = {
    "sunk_cost": "If you were starting today with nothing spent, would you still choose this?",
    "anchoring": "What would you think of this price or offer if you had never seen the first number?",
    "confirmation_bias": "What evidence would make you change your mind about this?",
    "loss_aversion": "What are you giving up by not taking the option with the bigger upside?",
}

NO_BIAS = {
    "bias_type": "none",
    "secondary_bias": None,
    "counter_scenario": "No clear bias was found in this entry, so there is nothing to object to. Your reasoning compares the trade-offs fairly.",
    "challenge_question": "What is the one thing that would make you change this decision?",
}

In [12]:
# Helper functions

In [13]:
def clean_text(text: str) -> str:
    """Swap long dashes for plain punctuation in model output."""
    return text.replace("\u2014", ",").replace("\u2013", "-")


def build_rebuttal_messages(entry: str, classification: dict) -> list:
    bias = classification["bias_type"]
    phrases = ", ".join(classification.get("trigger_phrases", [])) or "none"

    user = (f"Journal entry:\n{entry}\n\n"
            f"Detected bias: {bias}\n"
            f"Trigger phrases: {phrases}\n"
            f"Classifier reasoning: {classification.get('reasoning', '')}\n\n"
            f"Strategy for {bias}:\n{BIAS_STRATEGY[bias]}")

    secondary = classification.get("secondary_bias")
    if secondary in BIAS_STRATEGY:
        user += (f"\n\nA secondary bias was also found: {secondary}. "
                 f"Briefly touch on it using this strategy: {BIAS_STRATEGY[secondary]}")

    return [{"role": "system", "content": REBUTTAL_SYSTEM},
            {"role": "user", "content": user}]

In [14]:
# Rebuttal generator

In [15]:
def generate_rebuttal(entry: str, classification: dict, retries: int = 1) -> dict:
    bias = classification.get("bias_type", "none")

    # no bias found, nothing to argue against
    if bias not in BIAS_STRATEGY:
        return dict(NO_BIAS)

    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=build_rebuttal_messages(entry, classification),
                temperature=0.5,
                response_format={"type": "json_object"},
            )
            result = json.loads(response.choices[0].message.content)

            scenario = clean_text(result["counter_scenario"].strip())
            question = clean_text(result["challenge_question"].strip())
            if not scenario or not question:
                raise ValueError("Empty rebuttal")

            return {"bias_type": bias,
                    "secondary_bias": classification.get("secondary_bias"),
                    "counter_scenario": scenario,
                    "challenge_question": question}

        except (json.JSONDecodeError, ValueError, KeyError, AttributeError):
            if attempt == retries:
                return {"bias_type": bias,
                        "secondary_bias": classification.get("secondary_bias"),
                        "counter_scenario": "Could not generate a full rebuttal, showing the standard question for this bias.",
                        "challenge_question": FALLBACK_QUESTION[bias]}

In [16]:
# Test with hand-written classifications

In [17]:
samples = [
    ("I've already spent six months and 40k on this bootcamp. I hate it, but I can't quit now, that would mean all of it was wasted.",
     {"bias_type": "sunk_cost", "secondary_bias": None, "confidence_score": 0.93,
      "trigger_phrases": ["already spent six months and 40k", "can't quit now"],
      "reasoning": "The decision rests on past investment, not the bootcamp's future value."}),

    ("The laptop was originally 1,20,000 and now it's 85,000, so it's a massive deal and I should grab it.",
     {"bias_type": "anchoring", "secondary_bias": None, "confidence_score": 0.9,
      "trigger_phrases": ["originally 1,20,000", "it's a massive deal"],
      "reasoning": "The original price sets the reference point for value."}),

    ("I always knew my manager hates me. She skipped my update today, which proves it. The times she praised me were just her being polite.",
     {"bias_type": "confirmation_bias", "secondary_bias": None, "confidence_score": 0.94,
      "trigger_phrases": ["which proves it", "just her being polite"],
      "reasoning": "One ambiguous event is read as proof while the praise is explained away."}),

    ("The new job pays 40% more but I'd lose my seniority. I can't risk losing what I have, even if the upside is bigger.",
     {"bias_type": "loss_aversion", "secondary_bias": None, "confidence_score": 0.92,
      "trigger_phrases": ["can't risk losing what I have", "even if the upside is bigger"],
      "reasoning": "Fear of losing current status outweighs a larger admitted upside."}),

    ("I've put two years into this startup and can't walk away with nothing. Anyone who says it's failing just doesn't get our vision.",
     {"bias_type": "sunk_cost", "secondary_bias": "confirmation_bias", "confidence_score": 0.8,
      "trigger_phrases": ["put two years into this startup", "just doesn't get our vision"],
      "reasoning": "Main reason is past investment, and criticism is dismissed instead of evaluated."}),

    ("I made a table of rent, commute and lease flexibility for both flats, and the closer one wins since time matters most to me.",
     {"bias_type": "none", "secondary_bias": None, "confidence_score": 0.9,
      "trigger_phrases": [],
      "reasoning": "Trade-offs are compared against stated priorities."}),
]

for entry, classification in samples:
    out = generate_rebuttal(entry, classification)
    print(f"[{out['bias_type']}] secondary={out['secondary_bias']}")
    print(out["counter_scenario"])
    print("Q:", out["challenge_question"])
    print()

[sunk_cost] secondary=None
Imagine you could hit the reset button right now, with no six months behind you and no $40,000 already gone. Would you still enroll in this bootcamp if you only considered what lies ahead? Think about the extra weeks, the tuition you’d still have to pay, and the time you could spend on something you actually enjoy. The money and months you’ve already invested are gone whether you stay or leave, and staying now adds more cost to your schedule and wallet. Only the future benefits and expenses should guide your choice.
Q: If you started fresh today with no prior investment, would you sign up for the same bootcamp?

[anchoring] secondary=None
Imagine you walk into a store and see a laptop priced at 85,000 without any mention of its former price. You compare it to other models that cost around 90,000 to 1,00,000 and see that it's actually a modest discount, not a massive deal. If you had never heard the 1,20,000 figure, you might view the price as reasonable rathe

In [18]:
# Connect to the real classifier

In [19]:
entry = "I've already spent two years preparing for this exam, so I can't stop now even though I'm burnt out."

classification = classify_entry(entry)
print(json.dumps(classification, indent=2))

rebuttal = generate_rebuttal(entry, classification)
print()
print(json.dumps(rebuttal, indent=2))

{
  "bias_type": "sunk_cost",
  "secondary_bias": null,
  "confidence_score": 0.94,
  "trigger_phrases": [
    "already spent two years preparing for this exam",
    "can't stop now"
  ],
  "reasoning": "The decision to continue is justified solely by the time already invested, ignoring current burnout and future outcomes."
}

{
  "bias_type": "sunk_cost",
  "secondary_bias": null,
  "counter_scenario": "You have been preparing for this exam for two years and feel burnt out. Imagine you are starting fresh today with no time, money, or effort already invested. Would you still decide to commit to the same exam under those conditions? The two years you spent are already gone, and continuing now adds more stress and time that could be spent elsewhere. Think only about the future costs and benefits from this moment forward.",
  "challenge_question": "If you began today with zero investment, would you still choose to pursue this exam?"
}


## By Vishwa Chokshi

## Evaluation Dataset

In [20]:
evaluation_tests = [
    {
        "id": 1,
        "entry": "I have spent eight months preparing for this exam. I dislike the subject, but stopping now would waste all my effort.",
        "expected_bias": "sunk_cost"
    },
    {
        "id": 2,
        "entry": "I paid for a course last year and barely use it. I still want to continue only because I already paid the full fee.",
        "expected_bias": "sunk_cost"
    },
    {
        "id": 3,
        "entry": "Our project has lost customers for months, but we have invested two years in it, so we cannot stop now.",
        "expected_bias": "sunk_cost"
    },
    {
        "id": 4,
        "entry": "I am unhappy with this degree, but leaving after three years would make the time I spent feel pointless.",
        "expected_bias": "sunk_cost"
    },
    {
        "id": 5,
        "entry": "The original price was 9000 and the current price is 6500, so it must be a great deal.",
        "expected_bias": "anchoring"
    },
    {
        "id": 6,
        "entry": "My first salary offer was 5 LPA. The revised offer is 6 LPA, so I feel I should accept without checking market salaries.",
        "expected_bias": "anchoring"
    },
    {
        "id": 7,
        "entry": "The seller first asked for 80000 and then offered 70000. I think it is cheap because of the first price.",
        "expected_bias": "anchoring"
    },
    {
        "id": 8,
        "entry": "The first estimate for the repair was 20000, so the new estimate of 15000 seems reasonable even though I have not compared other shops.",
        "expected_bias": "anchoring"
    },
    {
        "id": 9,
        "entry": "I believe my teammate is careless. I only remember their mistakes and ignore the times they completed work correctly.",
        "expected_bias": "confirmation_bias"
    },
    {
        "id": 10,
        "entry": "I only read reviews that support buying this phone. The negative reviews are probably written by people who do not understand it.",
        "expected_bias": "confirmation_bias"
    },
    {
        "id": 11,
        "entry": "I am convinced my idea will succeed. I ignore criticism and keep collecting examples that agree with my opinion.",
        "expected_bias": "confirmation_bias"
    },
    {
        "id": 12,
        "entry": "I think the interview went badly, so I focus on one awkward answer and dismiss the positive feedback from the interviewer.",
        "expected_bias": "confirmation_bias"
    },
    {
        "id": 13,
        "entry": "The new job offers better pay and growth, but I am too afraid of losing my current stability to consider it.",
        "expected_bias": "loss_aversion"
    },
    {
        "id": 14,
        "entry": "Investing could give me a better return, but even a small possible loss worries me more than the potential gain.",
        "expected_bias": "loss_aversion"
    },
    {
        "id": 15,
        "entry": "I want to apply for the opportunity, but I keep thinking about what I might lose and ignore what I could gain.",
        "expected_bias": "loss_aversion"
    },
    {
        "id": 16,
        "entry": "The new role has strong benefits, but I would rather reject it than risk giving up the comfort of my current role.",
        "expected_bias": "loss_aversion"
    },
    {
        "id": 17,
        "entry": "I compared the costs, benefits, and travel times of three colleges and chose the one that best fits my priorities.",
        "expected_bias": "none"
    },
    {
        "id": 18,
        "entry": "I checked several independent reviews and prices before choosing the laptop that best matched my needs and budget.",
        "expected_bias": "none"
    },
    {
        "id": 19,
        "entry": "I listed the advantages and disadvantages of both job offers and chose the one that fits my long-term goals.",
        "expected_bias": "none"
    },
    {
        "id": 20,
        "entry": "I asked people with different opinions, reviewed the evidence on both sides, and then made my decision.",
        "expected_bias": "none"
    }
]

print("Total evaluation entries:", len(evaluation_tests))

for label in [
    "sunk_cost",
    "anchoring",
    "confirmation_bias",
    "loss_aversion",
    "none"
]:
    count = sum(
        item["expected_bias"] == label
        for item in evaluation_tests
    )
    print(f"{label}: {count}")

Total evaluation entries: 20
sunk_cost: 4
anchoring: 4
confirmation_bias: 4
loss_aversion: 4
none: 4


## Classification evaluation

In [21]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

LABELS = [
    "sunk_cost",
    "anchoring",
    "confirmation_bias",
    "loss_aversion",
    "none"
]

evaluation_results = []

for item in evaluation_tests:
    try:
        prediction = classify_entry(item["entry"])
        predicted_bias = prediction.get("bias_type", "none")

        if predicted_bias not in LABELS:
            predicted_bias = "none"

        evaluation_results.append({
            "id": item["id"],
            "expected_bias": item["expected_bias"],
            "predicted_bias": predicted_bias,
            "confidence": prediction.get("confidence_score", 0.0),
            "correct": predicted_bias == item["expected_bias"]
        })

        print(
            f"Entry {item['id']:02d} | "
            f"Expected: {item['expected_bias']} | "
            f"Predicted: {predicted_bias} | "
            f"{'PASS' if predicted_bias == item['expected_bias'] else 'FAIL'}"
        )

    except Exception as error:
        print(f"Entry {item['id']:02d} could not be evaluated: {error}")

results_df = pd.DataFrame(evaluation_results)

if len(results_df) == 0:
    print("No results were collected. Check the API key and notebook setup.")
else:
    correct_count = int(results_df["correct"].sum())
    total_count = len(results_df)
    accuracy = accuracy_score(
        results_df["expected_bias"],
        results_df["predicted_bias"]
    )

    print("\nCLASSIFICATION RESULTS")
    print(f"Entries evaluated: {total_count}/{len(evaluation_tests)}")
    print(f"Correct predictions: {correct_count}")
    print(f"Incorrect predictions: {total_count - correct_count}")
    print(f"Accuracy: {accuracy:.2%}")

    print("\nCLASSIFICATION REPORT")
    print(
        classification_report(
            results_df["expected_bias"],
            results_df["predicted_bias"],
            labels=LABELS,
            zero_division=0
        )
    )

    print("CONFUSION MATRIX")
    confusion_df = pd.DataFrame(
        confusion_matrix(
            results_df["expected_bias"],
            results_df["predicted_bias"],
            labels=LABELS
        ),
        index=[f"Actual: {label}" for label in LABELS],
        columns=[f"Predicted: {label}" for label in LABELS]
    )

    display(confusion_df)

    print("\nDETAILED RESULTS")
    display(results_df)

Entry 01 | Expected: sunk_cost | Predicted: sunk_cost | PASS
Entry 02 | Expected: sunk_cost | Predicted: sunk_cost | PASS
Entry 03 | Expected: sunk_cost | Predicted: sunk_cost | PASS
Entry 04 | Expected: sunk_cost | Predicted: sunk_cost | PASS
Entry 05 | Expected: anchoring | Predicted: anchoring | PASS
Entry 06 | Expected: anchoring | Predicted: anchoring | PASS
Entry 07 | Expected: anchoring | Predicted: anchoring | PASS
Entry 08 | Expected: anchoring | Predicted: anchoring | PASS
Entry 09 | Expected: confirmation_bias | Predicted: confirmation_bias | PASS
Entry 10 | Expected: confirmation_bias | Predicted: confirmation_bias | PASS
Entry 11 | Expected: confirmation_bias | Predicted: confirmation_bias | PASS
Entry 12 | Expected: confirmation_bias | Predicted: confirmation_bias | PASS
Entry 13 | Expected: loss_aversion | Predicted: loss_aversion | PASS
Entry 14 | Expected: loss_aversion | Predicted: loss_aversion | PASS
Entry 15 | Expected: loss_aversion | Predicted: loss_aversion | PA

,Predicted: sunk_cost,Predicted: anchoring,Predicted: confirmation_bias,Predicted: loss_aversion,Predicted: none
Actual: sunk_cost,4,0,0,0,0
Actual: anchoring,0,4,0,0,0
Actual: confirmation_bias,0,0,4,0,0
Actual: loss_aversion,0,0,0,4,0
Actual: none,0,0,0,0,4



DETAILED RESULTS


,id,expected_bias,predicted_bias,confidence,correct
0,1,sunk_cost,sunk_cost,0.92,True
1,2,sunk_cost,sunk_cost,0.93,True
2,3,sunk_cost,sunk_cost,0.94,True
3,4,sunk_cost,sunk_cost,0.89,True
4,5,anchoring,anchoring,0.92,True
5,6,anchoring,anchoring,0.92,True
6,7,anchoring,anchoring,0.92,True
7,8,anchoring,anchoring,0.90,True
8,9,confirmation_bias,confirmation_bias,0.93,True
9,10,confirmation_bias,confirmation_bias,0.93,True


## Rebuttal Quality Validation

In [22]:
BIAS_CHECK_TERMS = {
    "sunk_cost": [
        "starting today",
        "start from scratch",
        "already spent",
        "past investment",
        "future costs",
        "zero time",
        "zero money",
        "past cost"
    ],
    "anchoring": [
        "original price",
        "first number",
        "first offer",
        "reference point",
        "market rate",
        "market price",
        "compare other",
        "without seeing",
        "similar options"
    ],
    "confirmation_bias": [
        "opposing evidence",
        "contrary evidence",
        "evidence against",
        "change your mind",
        "different perspective",
        "alternative explanation",
        "disagree",
        "challenge your belief"
    ],
    "loss_aversion": [
        "potential gain",
        "potential loss",
        "upside",
        "downside",
        "give up",
        "risk",
        "benefit",
        "what you lose",
        "what you gain"
    ],
    "none": [
        "no clear bias",
        "compares the trade-offs",
        "balanced",
        "trade-offs fairly"
    ]
}


def check_rebuttal_quality(bias_type, rebuttal):
    """
    First-pass check for bias-specific reasoning.
    This is a keyword check, not a complete semantic evaluation.
    """

    if bias_type not in BIAS_CHECK_TERMS:
        return {
            "status": "REVIEW",
            "matched_terms": [],
            "reason": "Unknown bias label."
        }

    scenario = rebuttal.get("counter_scenario", "")
    question = rebuttal.get("challenge_question", "")

    combined_text = f"{scenario} {question}".lower()

    matched_terms = [
        term for term in BIAS_CHECK_TERMS[bias_type]
        if term in combined_text
    ]

    if not scenario.strip() or not question.strip():
        status = "FAIL"
        reason = "The rebuttal or challenge question is missing."

    elif bias_type == "none":
        status = "PASS" if len(matched_terms) >= 1 else "REVIEW"
        reason = (
            "No-bias response is consistent with the expected output."
            if status == "PASS"
            else "Check whether the response avoids inventing a bias."
        )

    elif len(matched_terms) >= 2:
        status = "PASS"
        reason = "Multiple bias-specific indicators were found."

    elif len(matched_terms) == 1:
        status = "REVIEW"
        reason = "Only one indicator was found. Read the response manually."

    else:
        status = "FAIL"
        reason = "No expected bias-specific indicators were found."

    return {
        "status": status,
        "matched_terms": matched_terms,
        "reason": reason
    }


print("Rebuttal quality checker is ready.")

Rebuttal quality checker is ready.


## Test rebuttal quality

In [23]:
sample_ids = [1, 5, 9, 13, 17]
rebuttal_results = []

for sample_id in sample_ids:
    item = next(
        test for test in evaluation_tests
        if test["id"] == sample_id
    )

    try:
        classification = classify_entry(item["entry"])
        rebuttal = generate_rebuttal(
            item["entry"],
            classification
        )

        claimed_bias = rebuttal.get(
            "bias_type",
            classification.get("bias_type", "none")
        )

        quality = check_rebuttal_quality(
            claimed_bias,
            rebuttal
        )

        rebuttal_results.append({
            "id": sample_id,
            "expected_bias": item["expected_bias"],
            "detected_bias": classification.get("bias_type"),
            "rebuttal_bias": claimed_bias,
            "status": quality["status"],
            "matched_terms": ", ".join(
                quality["matched_terms"]
            ),
            "reason": quality["reason"],
            "counter_scenario": rebuttal.get(
                "counter_scenario", ""
            ),
            "challenge_question": rebuttal.get(
                "challenge_question", ""
            )
        })

        print(f"\nEntry {sample_id}")
        print("Expected bias:", item["expected_bias"])
        print("Detected bias:", classification.get("bias_type"))
        print("Rebuttal status:", quality["status"])
        print("Matched terms:", quality["matched_terms"])
        print("Reason:", quality["reason"])
        print("Counter-scenario:", rebuttal.get("counter_scenario", ""))
        print("Challenge question:", rebuttal.get("challenge_question", ""))

    except Exception as error:
        print(f"Entry {sample_id} could not be checked: {error}")

rebuttal_df = pd.DataFrame(rebuttal_results)

if not rebuttal_df.empty:
    print("\nREBUTTAL QUALITY SUMMARY")
    print(rebuttal_df["status"].value_counts())
    display(rebuttal_df)


Entry 1
Expected bias: sunk_cost
Detected bias: sunk_cost
Rebuttal status: REVIEW
Matched terms: ['already spent']
Reason: Only one indicator was found. Read the response manually.
Counter-scenario: Imagine you could press a reset button right now and start the exam preparation from scratch, with no eight months behind you. Would you still commit to the same schedule and subject you now dislike, knowing only the future benefits and costs matter? Think about the extra weeks you will still need to study and the stress you will feel if you keep going, versus the possibility of redirecting that time toward something you enjoy. The time you have already spent is gone whether you continue or quit, so the decision rests on what you gain or lose from this point forward.
Challenge question: If you started today with zero investment, would you still choose to keep preparing for this exam?

Entry 5
Expected bias: anchoring
Detected bias: anchoring
Rebuttal status: PASS
Matched terms: ['original 

,id,expected_bias,detected_bias,rebuttal_bias,status,matched_terms,reason,counter_scenario,challenge_question
0,1,sunk_cost,sunk_cost,sunk_cost,REVIEW,already spent,Only one indicator was found. Read the respons...,Imagine you could press a reset button right n...,"If you started today with zero investment, wou..."
1,5,anchoring,anchoring,anchoring,PASS,"original price, market price",Multiple bias-specific indicators were found.,Imagine you walk into the store without ever h...,"Based on current market prices around $7,200, ..."
2,9,confirmation_bias,confirmation_bias,confirmation_bias,FAIL,,No expected bias-specific indicators were found.,"Imagine your team lead, who has examined every...",What single piece of evidence would make you a...
3,13,loss_aversion,loss_aversion,loss_aversion,FAIL,,No expected bias-specific indicators were found.,Imagine you stay at your current job and miss ...,Which outcome would bother you more in a year:...
4,17,none,none,none,PASS,"no clear bias, compares the trade-offs, trade-...",No-bias response is consistent with the expect...,"No clear bias was found in this entry, so ther...",What is the one thing that would make you chan...


## Save evaluation results

In [24]:
if not results_df.empty:
    results_df.to_csv(
        "classification_evaluation_results.csv",
        index=False
    )
    print("Classification results saved.")

if not rebuttal_df.empty:
    rebuttal_df.to_csv(
        "rebuttal_quality_results.csv",
        index=False
    )
    print("Rebuttal results saved.")

Classification results saved.
Rebuttal results saved.


## Generate evaluation report

In [25]:
from docx import Document
from docx.shared import Inches
from datetime import date

report = Document()

report.add_heading(
    "Objection, Your Honor: Evaluation and Quality Validation",
    level=0
)

report.add_paragraph("Prepared by: Vishwa Chokshi")
report.add_paragraph("Role: Person 3")
report.add_paragraph(f"Report date: {date.today().isoformat()}")

report.add_heading("1. Objective", level=1)
report.add_paragraph(
    "The objective of this work is to evaluate the cognitive bias "
    "classification engine using labeled journal entries and to "
    "check whether generated rebuttals address the bias they claim "
    "to challenge."
)

report.add_heading("2. Test Dataset", level=1)
report.add_paragraph(
    f"The test set contains {len(evaluation_tests)} journal entries "
    "covering sunk cost, anchoring, confirmation bias, loss aversion, "
    "and no clear bias. Each entry has an expected label assigned "
    "before running the classifier."
)

dataset_table = report.add_table(rows=1, cols=2)
dataset_table.style = "Table Grid"
dataset_table.rows[0].cells[0].text = "Expected label"
dataset_table.rows[0].cells[1].text = "Number of entries"

for label in LABELS:
    count = sum(
        item["expected_bias"] == label
        for item in evaluation_tests
    )
    cells = dataset_table.add_row().cells
    cells[0].text = label
    cells[1].text = str(count)

report.add_heading("3. Classification Results", level=1)

if not results_df.empty:
    accuracy = accuracy_score(
        results_df["expected_bias"],
        results_df["predicted_bias"]
    )

    report.add_paragraph(
        f"Entries evaluated: {len(results_df)} out of "
        f"{len(evaluation_tests)}"
    )
    report.add_paragraph(
        f"Correct predictions: {int(results_df['correct'].sum())}"
    )
    report.add_paragraph(
        f"Accuracy: {accuracy:.2%}"
    )

    report.add_paragraph(
        "The following table contains the classification report."
    )

    metrics = classification_report(
        results_df["expected_bias"],
        results_df["predicted_bias"],
        labels=LABELS,
        output_dict=True,
        zero_division=0
    )

    metric_table = report.add_table(rows=1, cols=5)
    metric_table.style = "Table Grid"

    headers = [
        "Label", "Precision", "Recall", "F1 score", "Support"
    ]
    for index, header in enumerate(headers):
        metric_table.rows[0].cells[index].text = header

    for label in LABELS:
        row = metric_table.add_row().cells
        row[0].text = label
        row[1].text = f"{metrics[label]['precision']:.3f}"
        row[2].text = f"{metrics[label]['recall']:.3f}"
        row[3].text = f"{metrics[label]['f1-score']:.3f}"
        row[4].text = str(int(metrics[label]["support"]))

    report.add_heading("Entry-wise predictions", level=2)

    prediction_table = report.add_table(rows=1, cols=4)
    prediction_table.style = "Table Grid"

    for index, header in enumerate(
        ["Entry ID", "Expected", "Predicted", "Correct"]
    ):
        prediction_table.rows[0].cells[index].text = header

    for _, result in results_df.iterrows():
        row = prediction_table.add_row().cells
        row[0].text = str(result["id"])
        row[1].text = str(result["expected_bias"])
        row[2].text = str(result["predicted_bias"])
        row[3].text = "Yes" if result["correct"] else "No"

else:
    report.add_paragraph(
        "Classification results were unavailable. Run the "
        "classification evaluation before generating the final report."
    )

report.add_heading("4. Rebuttal Quality Validation", level=1)

if not rebuttal_df.empty:
    report.add_paragraph(
        f"Rebuttals checked: {len(rebuttal_df)}"
    )

    for status in ["PASS", "REVIEW", "FAIL"]:
        count = int((rebuttal_df["status"] == status).sum())
        report.add_paragraph(f"{status}: {count}")

    rebuttal_table = report.add_table(rows=1, cols=4)
    rebuttal_table.style = "Table Grid"

    for index, header in enumerate(
        ["Entry ID", "Expected bias", "Status", "Reason"]
    ):
        rebuttal_table.rows[0].cells[index].text = header

    for _, result in rebuttal_df.iterrows():
        row = rebuttal_table.add_row().cells
        row[0].text = str(result["id"])
        row[1].text = str(result["expected_bias"])
        row[2].text = str(result["status"])
        row[3].text = str(result["reason"])

else:
    report.add_paragraph(
        "No rebuttal results were available. Run the rebuttal "
        "quality validation before finalizing this report."
    )

report.add_heading("5. Limitations", level=1)
report.add_paragraph(
    "The test dataset is small and manually labeled. Classification "
    "results depend on the quality of these labels and the examples "
    "selected. The rebuttal checker uses predefined keywords and "
    "cannot fully judge meaning, context, or logical relevance. "
    "Responses marked PASS should still be reviewed, and responses "
    "marked REVIEW or FAIL should be inspected before drawing "
    "conclusions. These results are an initial evaluation and do "
    "not establish general performance on all real-world journal entries."
)

report.add_heading("6. Conclusion", level=1)
report.add_paragraph(
    "This evaluation provides an initial measure of classification "
    "performance and a basic check for bias-specific rebuttals. "
    "The observed results can help identify labels that need "
    "improvement and responses that require further review."
)

report_path = "Evaluation_and_Validation_Report.docx"
report.save(report_path)

print(f"Report saved as: {report_path}")

Report saved as: Evaluation_and_Validation_Report.docx


## ----